# Linear Regression — Optimised with Learned Embeddings (MovieLens)

**Goal**: Find the best Linear Regression model for rating prediction
using the dense embedding features learned by the ANN notebook.

**Optimisation**: `RidgeCV` — sweeps regularisation strength `alpha`
automatically across 100 log-spaced values in a single fit.
No loops, no GridSearchCV overhead.

**Reads from**: `optimization_ANN\`
**Saves to**:   `optimization_LR\`


## Roadmap

```
STEP 1  — Imports & paths
STEP 2  — Load pre-split data
STEP 3  — Numerical preprocessing  (impute → log1p → scale)
STEP 4  — Load embeddings & build feature matrix
STEP 5  — Train Plain LR and Ridge (RidgeCV)
STEP 6  — Evaluate & compare the two
STEP 7  — Feature importance (Ridge coefficients)
STEP 8  — Save results to optimization_LR\
```


---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')


In [ ]:
from sklearn.linear_model  import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute        import SimpleImputer
from sklearn.metrics       import mean_squared_error, mean_absolute_error


In [ ]:
DATA_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\data"
EMB_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\embeddings\optimization_ANN"
OUT_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\optimization_LR"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Reading embeddings from : {EMB_DIR}")
print(f"Saving results to       : {OUT_DIR}")


In [ ]:
# Confirm embedding files exist before going any further
required = ['genre_embeddings.npy', 'lang_embeddings.npy',
            'genre_encoder.json',    'lang_encoder.json']

for fname in required:
    fpath  = os.path.join(EMB_DIR, fname)
    status = '✓' if os.path.exists(fpath) else '✗  MISSING — run ANN notebook first'
    print(f"  {status}  {fname}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(DATA_DIR + '\\train.csv')
df_val   = pd.read_csv(DATA_DIR + '\\val.csv')
df_test  = pd.read_csv(DATA_DIR + '\\test.csv')

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")


In [ ]:
TARGET    = 'rating'
CAT_GENRE = 'genres'
CAT_LANG  = 'original_language'

NUM_COLS = [
    'imdbRating', 'imdbVotes',
    'tmdbRating', 'tmdbVotes',
    'popularity', 'budget', 'runtime', 'revenue',
    'tag_count',  'avg_relevance', 'max_relevance'
]

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values


---
## Step 3 — Numerical Preprocessing

Same 3-step pipeline as the ANN notebook.
Always fit on train only — never on val or test.


In [ ]:
# Tag features: missing = no tag activity → fill with 0
for df_ in [df_train, df_val, df_test]:
    df_[['tag_count', 'avg_relevance', 'max_relevance']] =         df_[['tag_count', 'avg_relevance', 'max_relevance']].fillna(0)


In [ ]:
# Median imputation — fit on train, apply to val & test
imputer = SimpleImputer(strategy='median')
df_train[NUM_COLS] = imputer.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = imputer.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = imputer.transform(df_test[NUM_COLS])


In [ ]:
# log1p — compresses right-skewed distributions (votes, budget, revenue)
for df_ in [df_train, df_val, df_test]:
    df_[NUM_COLS] = np.log1p(df_[NUM_COLS].clip(lower=0))


In [ ]:
# StandardScaler — fit on train only
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = scaler.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = scaler.transform(df_test[NUM_COLS])

print("Numerical preprocessing done.")


---
## Step 4 — Load Embeddings & Build Feature Matrix

Load the weight matrices and vocab mappings saved by the ANN notebook.

For each row:
- **Language** → look up index in `lang_encoder.json` → fetch row from `lang_embeddings.npy`
- **Genres** → split by `|` → look up each token → mean-pool the genre vectors

Final matrix shape: `11 numeric + lang_emb_dim + genre_emb_dim` columns.


In [ ]:
# Load weight matrices
genre_weights = np.load(os.path.join(EMB_DIR, 'genre_embeddings.npy'))
lang_weights  = np.load(os.path.join(EMB_DIR, 'lang_embeddings.npy'))

print(f"Genre embedding matrix : {genre_weights.shape}  (n_genres × emb_dim)")
print(f"Lang  embedding matrix : {lang_weights.shape}   (n_langs  × emb_dim)")


In [ ]:
# Load vocab → index mappings
with open(os.path.join(EMB_DIR, 'genre_encoder.json')) as f:
    genre2idx = json.load(f)
with open(os.path.join(EMB_DIR, 'lang_encoder.json')) as f:
    lang2idx  = json.load(f)


In [ ]:
def lookup_lang(series):
    """Language string → embedding vector. Shape: (n_rows, lang_emb_dim)."""
    indices = series.fillna('<UNK>').map(lambda x: lang2idx.get(x, 0)).values
    return lang_weights[indices]

def lookup_genres(series):
    """Pipe-separated genre string → mean-pooled embedding. Shape: (n_rows, genre_emb_dim)."""
    result = []
    for g_str in series.fillna('<UNK>'):
        tokens  = g_str.split('|')
        indices = [genre2idx.get(t, 0) for t in tokens]
        vecs    = genre_weights[indices]       # (n_tokens, emb_dim)
        result.append(vecs.mean(axis=0))       # mean-pool → (emb_dim,)
    return np.array(result, dtype=np.float32)


In [ ]:
def build_feature_matrix(df_):
    """Concatenate numeric + lang embedding + genre embedding into one dense matrix."""
    num_part   = df_[NUM_COLS].values.astype(np.float32)
    lang_part  = lookup_lang(df_[CAT_LANG])
    genre_part = lookup_genres(df_[CAT_GENRE])
    return np.concatenate([num_part, lang_part, genre_part], axis=1)

X_train = build_feature_matrix(df_train)
X_val   = build_feature_matrix(df_val)
X_test  = build_feature_matrix(df_test)

genre_emb_dim = genre_weights.shape[1]
lang_emb_dim  = lang_weights.shape[1]

print(f"Feature matrix — train : {X_train.shape}")
print(f"  {len(NUM_COLS)} numeric  +  {lang_emb_dim} lang dims  +  {genre_emb_dim} genre dims")


In [ ]:
# Feature names — useful for coefficient inspection
feat_names = (NUM_COLS
              + [f'lang_emb_{i}' for i in range(lang_emb_dim)]
              + [f'genre_emb_{i}' for i in range(genre_emb_dim)])


---
## Step 5 — Train

We train two models:

| Model | What it does |
|-------|-------------|
| **Plain LR** | No regularisation — our direct baseline |
| **Ridge (RidgeCV)** | Adds L2 penalty. `RidgeCV` picks the best `alpha` from 100 log-spaced candidates using cross-validation in a single fit — no manual grid search needed |

Ridge is generally better than plain LR when features are correlated
(which embedding dimensions often are).


In [ ]:
# Plain Linear Regression — no regularisation
t0 = time.perf_counter()
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_train_time = time.perf_counter() - t0
print(f"Plain LR trained in {lr_train_time:.2f}s")


In [ ]:
# RidgeCV — sweeps 100 alpha values, picks the best via GCV
# Fitting the regularisation path analytically = no extra runtime vs a single fit
ALPHAS = np.logspace(-4, 4, 100)

t0 = time.perf_counter()
ridge_model = RidgeCV(alphas=ALPHAS, cv=5)
ridge_model.fit(X_train, y_train)
ridge_train_time = time.perf_counter() - t0

print(f"Ridge trained in {ridge_train_time:.2f}s")
print(f"Best alpha chosen by CV : {ridge_model.alpha_:.6f}")


---
## Step 6 — Evaluate & Compare

In [ ]:
def get_metrics(model, X, y):
    preds = model.predict(X)
    rmse  = np.sqrt(mean_squared_error(y, preds))
    mae   = mean_absolute_error(y, preds)
    return rmse, mae

results = []
for name, model, train_time in [
    ('Plain LR', lr_model,    lr_train_time),
    ('Ridge',    ridge_model, ridge_train_time)
]:
    tr_rmse,  tr_mae  = get_metrics(model, X_train, y_train)
    val_rmse, val_mae = get_metrics(model, X_val,   y_val)
    te_rmse,  te_mae  = get_metrics(model, X_test,  y_test)

    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  Train      RMSE: {tr_rmse:.4f}   MAE: {tr_mae:.4f}")
    print(f"  Validation RMSE: {val_rmse:.4f}   MAE: {val_mae:.4f}")
    print(f"  Test       RMSE: {te_rmse:.4f}   MAE: {te_mae:.4f}")
    print(f"  Train time : {train_time:.2f}s")
    print(f"  Train–Test gap : {te_rmse - tr_rmse:.4f}")

    results.append({
        'model'        : name,
        'train_rmse'   : round(tr_rmse,  4),
        'val_rmse'     : round(val_rmse, 4),
        'test_rmse'    : round(te_rmse,  4),
        'train_mae'    : round(tr_mae,   4),
        'val_mae'      : round(val_mae,  4),
        'test_mae'     : round(te_mae,   4),
        'train_time_s' : round(train_time, 2),
    })


In [ ]:
results_df = pd.DataFrame(results)
pd.set_option('display.float_format', '{:.4f}'.format)
print(results_df[['model','train_rmse','val_rmse','test_rmse',
                   'train_mae','val_mae','test_mae']].to_string(index=False))


---
## Step 7 — Feature Importance (Ridge Coefficients)

Larger |coefficient| = stronger linear effect on the predicted rating.
Positive = increases predicted rating, negative = decreases it.


In [ ]:
coefs = ridge_model.coef_.ravel()

imp_df = (pd.DataFrame({'feature': feat_names, 'coef': coefs})
            .assign(abs_coef=lambda x: x['coef'].abs())
            .sort_values('abs_coef', ascending=False)
            .head(15))

print("Top 15 features by |coefficient|:")
print(imp_df[['feature', 'coef']].to_string(index=False))


In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(imp_df['feature'][::-1], imp_df['coef'][::-1], color='steelblue')
plt.axvline(0, color='black', linewidth=0.6)
plt.xlabel('Ridge coefficient')
plt.title('Ridge — Top 15 features by |coefficient|')
plt.tight_layout()
plt.show()


---
## Step 8 — Save Results

In [ ]:
# Save full results table
results_df.to_csv(os.path.join(OUT_DIR, 'lr_results.csv'), index=False)
print("lr_results.csv saved.")


In [ ]:
# Save best model result as JSON for the final cross-model comparison
best_row = results_df.loc[results_df['test_rmse'].idxmin()].to_dict()
best_row['notebook'] = 'optimization_LR'

with open(os.path.join(OUT_DIR, 'lr_best_result.json'), 'w') as f:
    json.dump(best_row, f, indent=2)

print(f"Best model  : {best_row['model']}")
print(f"Test RMSE   : {best_row['test_rmse']:.4f}")
print(f"Test MAE    : {best_row['test_mae']:.4f}")


In [ ]:
# Save Ridge feature importances
imp_df[['feature','coef','abs_coef']].to_csv(
    os.path.join(OUT_DIR, 'lr_feature_importance.csv'), index=False)
print("lr_feature_importance.csv saved.")


In [ ]:
# Confirm output files
print("\nFiles in optimization_LR:")
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f:<40}  {size:,} bytes")
